# Tugas 10 — Deteksi Simbol Label Cuci Baju dengan YOLO11

**Zahra Faiza Fithriah · 18323002 · EB4011 Pembelajaran Mesin Lanjut**

Notebook ini membangun sistem *object detection* untuk 28 simbol petunjuk cuci pakaian (nomor kelas mengikuti angka merah di tabel dosen). Datanya adalah gabungan foto label baju yang dikumpulkan dan dianotasi oleh teman-teman sekelas.

Alurnya:

1. Setup & unduh dataset mentah dari Google Drive kelas
2. Menyeragamkan anotasi dari 21 sumber (format & ID kelas berbeda-beda)
3. Cek visual hasil pemetaan kelas
4. Menyusun folder dataset (per orang, gabungan, split train/val/test)
5. Eksplorasi data
6. Training YOLO11n dengan *transfer learning*
7. Evaluasi (mAP, precision, recall, confusion matrix)
8. 5-Fold Cross Validation
9. Uji pada gambar di luar dataset
10. Bonus: aplikasi deteksi lewat kamera HP

> Jalankan dengan GPU: **Runtime → Change runtime type → T4 GPU**. Semua cell bisa dijalankan berurutan dari atas.

## 1. Setup

In [ ]:
!pip -q install ultralytics gdown gradio

import os, re, json, csv, random, shutil, math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from ultralytics import YOLO
import torch

random.seed(42)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU")

BASE = Path("/content")
RAW_ZIPS = BASE / "raw_zips"      # zip mentah dari Drive kelas
RAW = BASE / "raw"                # hasil ekstrak per orang
NORM = BASE / "norm"              # hasil normalisasi per orang
FINAL = BASE / "dataset_final"    # folder final (per orang, gabungan, split)

### Daftar kelas

Nama kelas = angka merah di tabel dosen. Di file label YOLO, ID-nya dimulai dari 0, jadi **ID YOLO = nomor kelas − 1** (kelas "1" Hand Wash → ID 0, kelas "28" Dry in Shade → ID 27). Kolom `cara` dipakai nanti untuk menampilkan arti simbol di aplikasi.

In [ ]:
KELAS = [
    ("Hand Wash", "Hanya boleh dicuci dengan tangan"),
    ("Do Not Wash", "Tidak boleh dicuci"),
    ("Machine Wash", "Boleh dicuci dengan mesin cuci"),
    ("Machine Wash, Permanent Press", "Mesin cuci, siklus permanent press"),
    ("Machine Wash, Gentle or Delicate", "Mesin cuci, mode gentle/delicate"),
    ("Water Temperature 30°C", "Cuci dengan air suhu 30°C"),
    ("Water Temperature 40°C", "Cuci dengan air suhu 40°C"),
    ("Water Temperature 50°C", "Cuci dengan air suhu 50°C"),
    ("Water Temperature 60°C", "Cuci dengan air suhu 60°C"),
    ("Water Temperature 70°C", "Cuci dengan air suhu 70°C"),
    ("Water Temperature 95°C", "Cuci dengan air suhu 95°C"),
    ("Cold Water Temperature", "Cuci dengan air dingin"),
    ("Warm Water Temperature", "Cuci dengan air hangat"),
    ("Hot Water Temperature", "Cuci dengan air panas"),
    ("Bleach If Needed", "Boleh pakai pemutih"),
    ("Do Not Bleach", "Tidak boleh pakai pemutih"),
    ("Non-Chlorine Bleach If Needed", "Boleh pakai pemutih non-klorin"),
    ("Iron. Steam", "Boleh disetrika & steam, suhu berapa pun"),
    ("Do Not Iron", "Tidak boleh disetrika"),
    ("Iron With Max Temperature 110°C", "Setrika suhu maks. 110°C"),
    ("Iron With Max Temperature 180°C", "Setrika suhu maks. 150°C"),
    ("Iron With Max Temperature 200°C", "Setrika suhu maks. 200°C"),
    ("Do Not Steam", "Tidak boleh disetrika dengan setrika uap"),
    ("Dry", "Boleh dijemur"),
    ("Do Not Wring", "Tidak boleh diperas"),
    ("Do Not Dry", "Tidak boleh dikeringkan dengan mesin"),
    ("Dry Flat", "Keringkan dengan cara dibentangkan datar"),
    ("Dry in Shade", "Jemur di tempat teduh"),
]
NAMES = [str(i) for i in range(1, 29)]   # nama kelas = angka merah
FULLNAMES = [k[0] for k in KELAS]
pd.DataFrame([(i + 1, i, a, b) for i, (a, b) in enumerate(KELAS)],
             columns=["kelas", "id_yolo", "arti", "cara"]).set_index("kelas")

## 2. Unduh dataset kelas

Folder Drive berisi 21 zip (satu per mahasiswa) + `class.jpg` (tabel simbol dosen).

In [ ]:
DRIVE_FOLDER = "https://drive.google.com/drive/folders/1a20SMXJqTKp3frVutXMA4g9s_AdjoR39"
RAW_ZIPS.mkdir(exist_ok=True)
!gdown --folder "$DRIVE_FOLDER" -O "$RAW_ZIPS" -q

zips = sorted(RAW_ZIPS.rglob("*.zip"))
print(len(zips), "zip ditemukan")
shutil.rmtree(RAW, ignore_errors=True); RAW.mkdir()
for z in zips:
    nama = re.sub(r"-2026\d{4}T.*$", "", z.stem)   # buang akhiran timestamp dari Drive
    shutil.unpack_archive(z, RAW / nama)
CLASS_JPG = next(RAW_ZIPS.rglob("class.jpg"))
sorted(p.name for p in RAW.iterdir())

## 3. Menyeragamkan anotasi

Ini bagian yang paling makan waktu. Setiap orang mengekspor dataset dengan cara berbeda:

| Format | Contoh sumber | Cara membaca ID kelas |
|---|---|---|
| Export Ultralytics Platform (`.ndjson`) | 18323003, 18323009, 18323023, … | `class_names` di baris pertama `.ndjson` |
| Export Roboflow (`data.yaml`) | 18323006, 18323022, 18323026, 18322017 | daftar `names` di `data.yaml` |
| Hanya `.jpg` + `.txt`, ID 0-based | 18323019, 18322026, 18322004, … | ID = nomor − 1 |
| Hanya `.jpg` + `.txt`, ID = nomor kelas | 18323002, 18323008, 18323011 | ID = nomor kelas langsung |
| Tanpa metadata sama sekali | 18323016, 18323024 | dipetakan manual lewat cek visual (bagian 4) |

Artinya ID `0` bisa berarti simbol 1, 7, atau 12 tergantung siapa yang membuat. Kalau langsung digabung, model akan belajar dari label yang salah. Jadi semua sumber dipetakan ke **nomor kelas dosen** dulu, baru dikonversi ke ID YOLO.

Preprocessing yang dilakukan bersamaan:
- **Orientasi EXIF dibakukan** (`ImageOps.exif_transpose`). Foto HP sering menyimpan rotasi di metadata. Kalau tidak dibakukan, gambar bisa terbaca miring sementara box-nya tegak.
- **Semua format (jpeg/png/webp) diubah ke JPG** dan sisi terpanjang diperkecil ke maks. 1280 px. Ukuran dataset turun drastis (foto asli sampai 4032 px), tapi simbol kecil masih cukup jelas untuk training di `imgsz=640`.
- **Box dipotong ke batas gambar** dan box rusak (lebar/tinggi ≈ 0) dibuang.
- Gambar visualisasi bbox buatan teman (`*_bbox.jpg`, folder "Foto dengan bounding box class") **tidak ikut**.

In [ ]:
T17 = {  # label teks dari roboflow 18322017 -> nomor dosen
    "bleach_if_needed": 15, "do_not_bleach": 16, "do_not_iron": 19, "dry_flat": 27, "hand_wash": 1,
    "iron": 18, "iron_one_dot": 20, "iron_two_dots": 21, "machine_wash_delicate": 5,
    "machine_wash_permanent_press": 4, "non_chlorine_bleach": 17, "wash_30c": 6, "wash_40c": 7,
    "wash_cold_one_dot": 12,
}

def name_to_num(n):
    n = str(n).strip()
    if n.isdigit(): return int(n)
    if n in T17: return T17[n]
    if n in FULLNAMES: return FULLNAMES.index(n) + 1
    raise ValueError(f"nama kelas tidak dikenal: {n}")

# folder -> (tag, mode pembacaan .txt)
SOURCES = {
    "18323008_Tugas8MLL": ("18323008", "num"),
    "18323011_Foto_ label baju dan bounding box": ("18323011", "num"),
    "18323014_Dataset Washing Guide": ("18323014", "ndjson"),
    "18323016_FotoBaju": ("18323016", {0: 7, 1: 16, 2: 21, 3: 6, 4: 20, 5: 17}),           # hasil cek visual
    "18323019_MLL_Dataset": ("18323019", "zero"),
    "18323023_Tugas 8": ("18323023", "ndjson"),
    "Data Baju_Jerry Alexander Tjoa_18323026": ("18323026", "yaml"),
    "Dataset MLL": ("DatasetMLL_nathan-jeremia", "num"),
    "Dataset Tugas 8_MLL_18323002_Zahra Faiza Fithriah": ("18323002_Zahra", "num"),
    "Dataset Tugas 8_Pembelajaran Mesin Lanjut_18323024_Felicia Kannitha":
        ("18323024", {0: 6, 1: 16, 2: 19, 3: 7, 4: 22, 5: 20, 6: 17, 7: 1, 8: 21}),          # hasil cek visual
    "EB4011_Tugas 8_18323006": ("18323006", "yaml"),
    "Laundry Dataset.v1i.yolov8": ("18323022", "yaml"),
    "MLL_Tugas 8_18323009": ("18323009", "ndjson"),
    "TUGAS 8_18322017": ("18322017", "yaml"),
    "Tugas 8 MLL_18323025": ("18323025", "zero"),
    "Tugas 8 Nurbaedah Awaliyah (18322027)": ("18322027", "yaml"),
    "Tugas 8_18322026": ("18322026", "zero"),
    "Tugas MLL 08_18323003": ("18323003", "ndjson"),
    "Tugas_Dataset_Label_Hafidz Y Patty_18322004": ("18322004", "zero"),
    "dataset18323015": ("18323015", "ndjson"),
    "laundry guide": ("laundry-guide", "zero"),
}
# salah label yang ketemu saat cek visual: {tag: {nomor_lama: nomor_benar}}
FIX = {"18323026": {22: 7}, "18323003": {28: 7}, "18322026": {12: 20}}
MAX_SIDE = 1280
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}
SKIP_TXT = {"readme.txt", "classes.txt", "readme.roboflow.txt"}
key = lambda s: re.sub(r"\s+", " ", s).strip().lower()

In [ ]:
import yaml

def baca_sumber(folder, tag, mode):
    src = RAW / folder
    imgs = [p for p in src.rglob("*") if p.suffix.lower() in IMG_EXT
            and "_bbox" not in p.stem.lower() and "bounding box class" not in str(p).lower()]
    txts = {key(p.stem): p for p in src.rglob("*.txt") if p.name.lower() not in SKIP_TXT}

    nd = {}   # anotasi dari .ndjson (paling bisa dipercaya karena membawa class_names)
    for f in src.rglob("*.ndjson"):
        rows = [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
        cn = {int(k): name_to_num(v) for k, v in rows[0]["class_names"].items()}
        for r in rows[1:]:
            if r.get("type") == "image":
                b = r.get("annotations", {}).get("boxes", [])
                nd[key(Path(r["file"]).stem)] = (r["width"], r["height"], [(cn[int(x[0])], *x[1:5]) for x in b])

    ymap = None
    if (ys := list(src.rglob("data.yaml"))):
        names = yaml.safe_load(ys[0].read_text())["names"]
        ymap = {i: name_to_num(n) for i, n in enumerate(names)}

    def to_num(i):
        if mode == "zero": return i + 1
        if mode == "num": return i
        if mode == "yaml": return ymap[i]
        if isinstance(mode, dict): return mode[i]

    out = NORM / tag
    (out / "images").mkdir(parents=True, exist_ok=True); (out / "labels").mkdir(parents=True, exist_ok=True)
    n_img = n_box = 0
    for ip in sorted(imgs):
        k = key(ip.stem)
        im0 = Image.open(ip); im = ImageOps.exif_transpose(im0).convert("RGB")
        if k in nd:
            W, H, boxes = nd[k]
            if (W, H) != im.size and (W, H) == im0.size:   # platform memakai orientasi mentah
                im = im0.convert("RGB")
        elif k in txts and mode != "ndjson":
            boxes = [(to_num(int(float(p[0]))), *map(float, p[1:]))
                     for p in (l.split() for l in txts[k].read_text().splitlines()) if len(p) == 5]
        else:
            continue
        rows = []
        for c, cx, cy, w, h in boxes:
            c = FIX.get(tag, {}).get(c, c)
            x1, y1 = max(0, cx - w / 2), max(0, cy - h / 2)
            x2, y2 = min(1, cx + w / 2), min(1, cy + h / 2)
            if x2 - x1 > 1e-3 and y2 - y1 > 1e-3 and 1 <= c <= 28:
                rows.append(f"{c-1} {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {x2-x1:.6f} {y2-y1:.6f}")
        if not rows:
            continue
        s = MAX_SIDE / max(im.size)
        if s < 1:
            im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
        name = f"{tag}__{re.sub(r'[^A-Za-z0-9_-]+', '_', ip.stem)}"
        im.save(out / "images" / f"{name}.jpg", quality=92)
        (out / "labels" / f"{name}.txt").write_text("\n".join(rows) + "\n")
        n_img += 1; n_box += len(rows)
    return n_img, n_box

shutil.rmtree(NORM, ignore_errors=True)
ringkas = []
for folder, (tag, mode) in SOURCES.items():
    ni, nb_ = baca_sumber(folder, tag, mode)
    ringkas.append((tag, mode if isinstance(mode, str) else "cek visual", ni, nb_))
df_src = pd.DataFrame(ringkas, columns=["sumber", "cara baca ID", "gambar", "box"])
print("Total:", df_src.gambar.sum(), "gambar,", df_src.box.sum(), "box")
df_src

## 4. Cek visual pemetaan kelas

Untuk memastikan pemetaannya benar, setiap box dipotong dan dijejerkan per kelas di samping ikon referensi dari `class.jpg`. Kalau satu baris isinya tidak mirip ikon di kirinya, berarti pemetaannya salah.

Cara ini juga yang dipakai untuk mencari pemetaan dua sumber tanpa metadata (18323016 dan 18323024), dan untuk menemukan beberapa salah label kecil yang dikoreksi lewat `FIX`.

In [ ]:
ref = Image.open(CLASS_JPG).convert("RGB"); RW, RH = ref.size
def ikon(num, T=80):
    r = (num - 1) // 2; x0 = 0.024 * RW if num % 2 else 0.503 * RW; cw = 0.159 * RW
    top, bot = 0.0375 * RH, 0.988 * RH; rh = (bot - top) / 14
    return ref.crop((int(x0 + .05 * cw), int(top + r * rh + .1 * rh), int(x0 + cw), int(top + (r + 1) * rh))).resize((T, T))

def montage(tag, ncol=7, T=80):
    src = NORM / tag; crops = defaultdict(list)
    for lp in sorted((src / "labels").glob("*.txt")):
        im = Image.open(src / "images" / f"{lp.stem}.jpg")
        for l in lp.read_text().splitlines():
            c, cx, cy, w, h = l.split(); c = int(c); cx, cy, w, h = map(float, (cx, cy, w, h))
            crops[c].append(im.crop((int((cx-w/2)*im.width), int((cy-h/2)*im.height),
                                     int((cx+w/2)*im.width), int((cy+h/2)*im.height))))
    rows = sorted(crops)
    fig, axes = plt.subplots(len(rows), ncol + 1, figsize=(1.3 * (ncol + 1), 1.3 * len(rows)))
    axes = np.atleast_2d(axes)
    for i, c in enumerate(rows):
        axes[i, 0].imshow(ikon(c + 1)); axes[i, 0].set_title(f"kelas {c+1} (n={len(crops[c])})", fontsize=8, color="crimson")
        for j, cr in enumerate(crops[c][:ncol]):
            axes[i, j + 1].imshow(cr)
    for a in axes.ravel(): a.axis("off")
    fig.suptitle(tag, fontsize=11); plt.tight_layout(); plt.show()

for tag in ["18323016", "18323024", "18323002_Zahra"]:
    montage(tag)

In [ ]:
# lihat juga sumber lain kalau perlu, misalnya:
# for tag in sorted(p.name for p in NORM.iterdir()): montage(tag)

### Catatan: dataset 18323015 tidak dipakai untuk training

Dataset ini cuma punya **1 box per foto**, padahal di fotonya terlihat 3–5 simbol. Simbol yang tidak diberi box akan dianggap *background* oleh YOLO. Model jadi justru diajari untuk **tidak** mendeteksi simbol yang sebenarnya ada. Jadi foto-foto ini tetap disimpan di folder per orang, tapi tidak ikut dataset gabungan.

In [ ]:
def tampil_bbox(img_path, lbl_path, ax=None, warna=(255, 0, 0)):
    im = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    for l in Path(lbl_path).read_text().splitlines():
        c, cx, cy, bw, bh = map(float, l.split())
        x1, y1, x2, y2 = int((cx-bw/2)*w), int((cy-bh/2)*h), int((cx+bw/2)*w), int((cy+bh/2)*h)
        cv2.rectangle(im, (x1, y1), (x2, y2), warna, max(2, w // 250))
        cv2.putText(im, str(int(c) + 1), (x1, max(15, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, w / 900, warna, max(2, w // 400))
    (ax or plt).imshow(im)
    if ax: ax.axis("off")

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, p in zip(axes, sorted((NORM / "18323015/images").glob("*.jpg"))[:4]):
    tampil_bbox(p, NORM / "18323015/labels" / f"{p.stem}.txt", ax)
plt.suptitle("18323015: hanya satu simbol yang dianotasi per foto"); plt.show()

## 5. Menyusun folder dataset

```
dataset_final/
├── 1_dataset_per_orang/          # hasil normalisasi, satu folder per mahasiswa
│   └── <sumber>/images, labels
├── 2_dataset_gabungan/           # semua sumber (kecuali 18323015) jadi satu
│   ├── images/  labels/
│   └── data.yaml
└── 3_dataset_yolo_split/         # yang dipakai training
    ├── images/{train,val,test}
    ├── labels/{train,val,test}
    └── data.yaml
```

Alasannya:
- **Per orang** supaya jejak asal data tetap ada. Kalau ada label salah, gampang dilacak dan diperbaiki di sumbernya.
- **Gabungan** sebagai satu kolam data bersih, dipakai juga untuk K-Fold.
- **Split** mengikuti struktur yang diminta Ultralytics (`images/` dan `labels/` sejajar, nama file sama). Pembagiannya 70/15/15 dan dilakukan **per sumber** (stratified by photographer), sehingga setiap orang punya foto di train, val, dan test. Kalau diacak global, bisa saja semua foto satu orang (kamera, pencahayaan, jenis baju yang sama) jatuh ke test. Hasil evaluasinya jadi bias.
- Nama file diberi awalan sumber (`18323002_Zahra__label_05.jpg`) supaya tidak ada nama bentrok antar orang (banyak yang pakai `1.jpg`, `label_01.jpg`, dst.).

In [ ]:
EXCLUDE = {"18323015"}
shutil.rmtree(FINAL, ignore_errors=True)
PER, GAB, SPL = FINAL / "1_dataset_per_orang", FINAL / "2_dataset_gabungan", FINAL / "3_dataset_yolo_split"
shutil.copytree(NORM, PER)
for tag in sorted(p.name for p in NORM.iterdir() if p.name not in EXCLUDE):
    for sub in ("images", "labels"):
        (GAB / sub).mkdir(parents=True, exist_ok=True)
        for f in (NORM / tag / sub).iterdir(): shutil.copy(f, GAB / sub / f.name)

random.seed(42); split_of = {}
for tag in sorted(p.name for p in NORM.iterdir() if p.name not in EXCLUDE):
    stems = sorted(p.stem for p in (NORM / tag / "labels").glob("*.txt")); random.shuffle(stems)
    k = max(1, round(0.15 * len(stems)))
    for i, s in enumerate(stems):
        split_of[s] = "test" if i < k else ("val" if i < 2 * k else "train")
for s, sp in split_of.items():
    for sub, ext in (("images", ".jpg"), ("labels", ".txt")):
        (SPL / sub / sp).mkdir(parents=True, exist_ok=True)
        shutil.copy(GAB / sub / f"{s}{ext}", SPL / sub / sp / f"{s}{ext}")

(SPL / "data.yaml").write_text(f"path: {SPL}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnc: 28\nnames: {json.dumps(NAMES)}\n")
(GAB / "data.yaml").write_text(f"path: {GAB}\ntrain: images\nval: images\n\nnc: 28\nnames: {json.dumps(NAMES)}\n")
DATA = str(SPL / "data.yaml")
print(Counter(split_of.values()))

## 6. Eksplorasi data

In [ ]:
cnt = {sp: Counter() for sp in ("train", "val", "test")}
for s, sp in split_of.items():
    for l in (GAB / "labels" / f"{s}.txt").read_text().splitlines():
        cnt[sp][int(l.split()[0]) + 1] += 1
dist = pd.DataFrame({sp: [cnt[sp][c] for c in range(1, 29)] for sp in cnt}, index=range(1, 29))

ax = dist.plot.bar(stacked=True, figsize=(13, 4), color=["#4C72B0", "#DD8452", "#55A868"], width=.8)
ax.set_xlabel("kelas"); ax.set_ylabel("jumlah box"); ax.set_title("Distribusi box per kelas")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
print("Kelas tanpa contoh sama sekali:", [c for c in range(1, 29) if dist.loc[c].sum() == 0])

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for ax, s in zip(axes.ravel(), random.sample(sorted(split_of), 8)):
    tampil_bbox(GAB / "images" / f"{s}.jpg", GAB / "labels" / f"{s}.txt", ax); ax.set_title(s[:28], fontsize=8)
plt.tight_layout(); plt.show()

## 7. Training YOLO11n (transfer learning)

**Kenapa YOLO11n?** Ini detektor satu tahap (*one-stage*): lokasi box dan kelas diprediksi sekaligus dalam satu kali lewat jaringan, jadi cepat dan cocok untuk dipakai *real-time* di HP. Versi *nano* (≈2,6 juta parameter) dipilih karena datanya kecil (≈200 foto). Model yang lebih besar akan lebih mudah *overfitting* dan lebih lambat. Ini juga model yang dipakai di contoh notebook kuliah.

**Strategi transfer learning:** bobot awal diambil dari `yolo11n.pt` yang sudah dilatih di COCO (80 kelas, ±118 ribu gambar). *Backbone*-nya sudah bisa mengenali garis, sudut, dan bentuk dasar, dan itu persis yang membentuk simbol label cuci. *Head* deteksi otomatis diganti untuk 28 kelas, lalu seluruh jaringan di-*fine-tune* dengan *learning rate* kecil plus *warmup*. Dengan data sesedikit ini, training dari nol hampir pasti gagal konvergen.

Pengaturan penting:
- `imgsz=640`: simbol sering cuma 3–8% lebar foto, jadi resolusi terlalu kecil (mis. 320) akan menghapus detailnya (titik suhu setrika, angka 30/40).
- `fliplr=0`: flip horizontal **dimatikan**. Simbol label tidak pernah tercetak terbalik, jadi flip cuma membuat contoh yang tidak realistis.
- `degrees=10`: rotasi kecil, karena foto label sering agak miring.
- Augmentasi lain (mosaic, HSV, scale, translate) memakai default Ultralytics.

In [ ]:
COMMON = dict(imgsz=640, batch=16, device=DEVICE, cache="ram", fliplr=0.0, degrees=10, seed=42,
              deterministic=True, exist_ok=True, project="/content/runs")

model = YOLO("yolo11n.pt")             # bobot pretrained COCO
model.train(data=DATA, epochs=40, patience=15, name="main", **COMMON)

In [ ]:
from IPython.display import Image as IImage, display
display(IImage("/content/runs/main/results.png", width=1000))

## 8. Evaluasi

Metrik yang dipakai:
- **mAP@0.5**: metrik utama. Rata-rata *Average Precision* semua kelas, dengan prediksi dihitung benar kalau IoU ≥ 0,5 dan kelasnya cocok. Satu angka ini sudah merangkum ketepatan lokasi dan klasifikasi, di semua ambang *confidence*.
- **mAP@0.5:0.95**: lebih ketat, rata-rata dari IoU 0,5 sampai 0,95. Menunjukkan seberapa presisi box-nya.
- **Precision & Recall**: berapa persen deteksi yang benar, dan berapa persen simbol yang berhasil ditemukan. Untuk aplikasi ini *recall* penting, karena simbol yang terlewat berarti instruksi cuci yang hilang.
- **Confusion matrix**: untuk melihat kelas mana yang tertukar (mis. setrika 1 titik vs 2 titik).

Accuracy biasa tidak dipakai karena di object detection tidak ada jumlah "negatif" yang jelas. Setiap posisi di gambar yang tidak berisi simbol adalah negatif, jadi angkanya tidak bermakna.

In [ ]:
best = YOLO("/content/runs/main/weights/best.pt")
hasil = {}
for sp in ("val", "test"):
    m = best.val(data=DATA, split=sp, imgsz=640, device=DEVICE, project="/content/runs", name=f"main_{sp}", exist_ok=True)
    hasil[sp] = dict(precision=m.box.mp, recall=m.box.mr, mAP50=m.box.map50, mAP50_95=m.box.map)
    if sp == "test": m_test = m
pd.DataFrame(hasil).T.round(3)

In [ ]:
rows = []
for i, c in enumerate(m_test.box.ap_class_index):
    p, r, ap50, ap = m_test.box.class_result(i)
    rows.append((int(c) + 1, FULLNAMES[int(c)], dist.loc[int(c) + 1, "train"], p, r, ap50, ap))
pd.DataFrame(rows, columns=["kelas", "arti", "box train", "P", "R", "AP50", "AP50-95"]).round(3).set_index("kelas")

In [ ]:
display(IImage("/content/runs/main_test/confusion_matrix_normalized.png", width=850))

### Ground truth vs prediksi (data test)

In [ ]:
test_imgs = sorted((SPL / "images/test").glob("*.jpg"))
pilih = random.Random(1).sample(test_imgs, 4)
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for j, p in enumerate(pilih):
    tampil_bbox(p, SPL / "labels/test" / f"{p.stem}.txt", axes[0, j], (0, 200, 0)); axes[0, j].set_title("ground truth", fontsize=9)
    r = best.predict(str(p), imgsz=640, conf=0.25, verbose=False)[0]
    axes[1, j].imshow(r.plot(line_width=2)[:, :, ::-1]); axes[1, j].axis("off"); axes[1, j].set_title("prediksi", fontsize=9)
plt.tight_layout(); plt.show()

## 9. 5-Fold Cross Validation

Dengan data sekecil ini, satu kali split bisa "beruntung" atau "sial" tergantung foto mana yang masuk validasi. K-Fold memberi gambaran yang lebih jujur. Data train+val (174 foto) dibagi jadi 5 bagian. Model dilatih 5 kali, tiap kali 1 bagian jadi validasi dan 4 lainnya untuk training. Data test tetap disisihkan dan tidak pernah dipakai di K-Fold.

Yang dilihat: rata-rata dan simpangan baku mAP antar fold. Kalau simpangan bakunya besar, performa model masih sangat bergantung pada foto mana yang dipakai.

In [ ]:
K = 5
pool = sorted(p.stem for sp in ("train", "val") for p in (SPL / "images" / sp).glob("*.jpg"))
src_of = {s: sp for sp in ("train", "val") for s in (p.stem for p in (SPL / "images" / sp).glob("*.jpg"))}
random.Random(42).shuffle(pool)
folds = [pool[i::K] for i in range(K)]

kres = []
for k in range(K):
    fd = Path(f"/content/kfold/fold{k+1}"); shutil.rmtree(fd, ignore_errors=True)
    for sp, stems in (("val", folds[k]), ("train", [s for j in range(K) if j != k for s in folds[j]])):
        for sub, ext in (("images", ".jpg"), ("labels", ".txt")):
            (fd / sub / sp).mkdir(parents=True, exist_ok=True)
            for s in stems: (fd / sub / sp / f"{s}{ext}").symlink_to(SPL / sub / src_of[s] / f"{s}{ext}")
    (fd / "data.yaml").write_text(f"path: {fd}\ntrain: images/train\nval: images/val\nnc: 28\nnames: {json.dumps(NAMES)}\n")
    YOLO("yolo11n.pt").train(data=str(fd / "data.yaml"), epochs=15, patience=15, name=f"fold{k+1}", plots=False, **COMMON)
    r = YOLO(f"/content/runs/fold{k+1}/weights/best.pt").val(data=str(fd / "data.yaml"), split="val", imgsz=640,
                                                             device=DEVICE, plots=False, verbose=False)
    kres.append(dict(fold=k + 1, n_train=len(pool) - len(folds[k]), n_val=len(folds[k]),
                     precision=r.box.mp, recall=r.box.mr, mAP50=r.box.map50, mAP50_95=r.box.map))

df_k = pd.DataFrame(kres).set_index("fold")
df_k.loc["rata-rata"] = df_k.mean(); df_k.loc["std"] = df_k.iloc[:K].std()
df_k.round(3)

## 10. Uji pada gambar di luar dataset

Model dicoba ke gambar yang **tidak pernah** ada di dataset mana pun:
1. `class.jpg`, tabel simbol dosen. Simbolnya berupa ikon digital yang bersih, bukan foto kain. Ini menguji apakah model mengenali bentuk simbolnya atau cuma tekstur label kain.
2. Foto label baju baru (upload sendiri lewat cell di bawah).

In [ ]:
r = best.predict(str(CLASS_JPG), imgsz=1280, conf=0.25, verbose=False)[0]
plt.figure(figsize=(9, 13)); plt.imshow(r.plot(line_width=2)[:, :, ::-1]); plt.axis("off"); plt.show()
print("Jumlah deteksi:", len(r.boxes))

In [ ]:
from google.colab import files
up = files.upload()            # pilih 1 atau beberapa foto label baju
for nama in up:
    r = best.predict(nama, imgsz=640, conf=0.25, verbose=False)[0]
    plt.figure(figsize=(7, 9)); plt.imshow(r.plot(line_width=2)[:, :, ::-1]); plt.axis("off"); plt.title(nama); plt.show()
    for b in sorted(r.boxes, key=lambda b: float(b.xyxy[0][0])):
        c = int(b.cls); print(f"  [{c+1:2d}] {FULLNAMES[c]:<34s} conf {float(b.conf):.2f}  → {KELAS[c][1]}")

## 11. Bonus: aplikasi deteksi lewat kamera HP

Cell di bawah menjalankan aplikasi web kecil (Gradio). Setelah dijalankan, muncul link `https://xxxx.gradio.live`. Buka link itu di HP, pilih ikon kamera, foto label baju, dan aplikasi akan menandai simbolnya sekaligus menjelaskan cara merawat pakaiannya dalam bahasa Indonesia.

Link aktif selama notebook masih berjalan (maks. 72 jam).

Selain itu, model yang sama juga sudah dijalankan sebagai halaman web mandiri yang tidak butuh Colab: model diekspor ke ONNX dan dijalankan di browser HP dengan ONNX Runtime Web. Link: https://USERNAME.github.io/label-cuci/

In [ ]:
import gradio as gr

def deteksi(img, conf):
    if img is None:
        return None, "Belum ada gambar."
    r = best.predict(img[:, :, ::-1], imgsz=640, conf=conf, verbose=False)[0]   # Gradio memberi RGB, YOLO mengharap BGR
    out = r.plot(line_width=3)[:, :, ::-1]
    if len(r.boxes) == 0:
        return out, "Tidak ada simbol terdeteksi. Coba dekatkan kamera ke label dan pastikan fokus."
    baris, sudah = [], set()
    for b in sorted(r.boxes, key=lambda b: float(b.xyxy[0][0])):
        c = int(b.cls)
        if c in sudah: continue
        sudah.add(c)
        baris.append(f"| **{c+1}** | {FULLNAMES[c]} | {KELAS[c][1]} | {float(b.conf):.0%} |")
    return out, "| No | Simbol | Cara perawatan | Yakin |\n|---|---|---|---|\n" + "\n".join(baris)

with gr.Blocks(title="Baca Label Cuci", theme=gr.themes.Soft()) as app:
    gr.Markdown("## Baca Label Cuci\nFoto label pakaian, nanti simbol perawatannya dibacakan.")
    with gr.Row():
        with gr.Column():
            inp = gr.Image(sources=["webcam", "upload"], type="numpy", label="Foto label")
            conf = gr.Slider(0.1, 0.9, value=0.3, step=0.05, label="Ambang keyakinan")
            btn = gr.Button("Baca simbol", variant="primary")
        with gr.Column():
            out_img = gr.Image(label="Hasil deteksi")
            out_txt = gr.Markdown()
    btn.click(deteksi, [inp, conf], [out_img, out_txt])
    inp.change(deteksi, [inp, conf], [out_img, out_txt])

app.launch(share=True, debug=False)

## 12. Simpan model & dataset

In [ ]:
EXP = Path("/content/export"); EXP.mkdir(exist_ok=True)
shutil.copy("/content/runs/main/weights/best.pt", EXP / "yolo11n_label_cuci_best.pt")
best.export(format="onnx", imgsz=640)                    # untuk dipakai di aplikasi lain / web
shutil.copy("/content/runs/main/weights/best.onnx", EXP / "yolo11n_label_cuci_best.onnx")
shutil.make_archive("/content/dataset_final", "zip", FINAL)
print(sorted(p.name for p in EXP.iterdir()))
# files.download("/content/export/yolo11n_label_cuci_best.pt")